# <a id='toc1_'></a>[Data check notebook](#toc0_)

**Table of contents**<a id='toc0_'></a>    
- [Data check notebook](#toc1_)    
    - [Config](#toc1_1_1_)    
    - [Import dependencies and load data](#toc1_1_2_)    
    - [Distribution of patients by age category](#toc1_1_3_)    
    - [Distribution Age at last follow-up](#toc1_1_4_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

### <a id='toc1_1_1_'></a>[Config](#toc0_)

In [ ]:
# logging setup
from puv.logging_setup import setup
setup()  # or DEBUG if you want more detail

import logging
logger = logging.getLogger(__name__)
logger.info("Logging initialized.")

In [ ]:
# Environment setup
from pathlib import Path

BASE = Path('/workspaces/CODESPACE/').resolve()

# Define paths for data and analysis directories & ensure key folders exist
PREP = BASE / "data" / "prep"

for p in [PREP]:
    p.mkdir(parents=True, exist_ok=True)

logger.info(f"Project base set to: {BASE}")

### <a id='toc1_1_2_'></a>[Import dependencies and load data](#toc0_)

In [ ]:
# Import dependencies
import pandas as pd
import numpy as np
import seaborn
import matplotlib.pyplot as plt

# load data
df_all = pd.read_excel(PREP / 'data_all.xlsx')

# Uncomment for look at the data info
# df_all.info()

plt.rcParams.update({
    "font.size": 12, "axes.titlesize": 15, "axes.labelsize": 13,
    "xtick.labelsize": 12, "ytick.labelsize": 12, "legend.fontsize": 12,
})


### <a id='toc1_1_3_'></a>[Distribution of patients by age category](#toc0_)

In [ ]:
# Count members of the categories of interest based on the 'TUR age (first)' column
df_dist_age = df_all[['TUR age (first)', 'Patient number']].copy()

# create categories for the TUR age (first)
df_dist_age['Category'] = df_dist_age['TUR age (first)'].apply(
    lambda x: 'Neonate' if x <= 1 else 'Infant' if x <= 12 else 'Child'
)

# Create value counts
counts = df_dist_age['Category'].value_counts()

# Create bar plot
ax = counts.plot(kind='bar', color="skyblue", edgecolor='black', figsize=(4, 3))
plt.title('Distribution of Patients by Age Category')
plt.ylim(0, counts.max() + 10)
plt.xlabel('Age Category')
plt.ylabel('Number of Patients')
plt.xticks(rotation=45)

# Add value labels on top of each bar
for i, v in enumerate(counts):
    ax.text(i, v, str(v), ha='center', va='bottom')

plt.show()

### <a id='toc1_1_4_'></a>[Distribution of Age at last follow-up](#toc0_)

In [ ]:
# retain only patient data pertaining to infant -- i.e., less than 12 months old and not nan
df_age_lf = (
    df_all
    .loc[df_all['TUR age (first)'] <= 12]
    .reset_index(drop=True)
    .copy()
)

# Coerce to numeric, turn non-numeric into NaN
df_age_lf['Age last follow up'] = pd.to_numeric(df_age_lf['Age last follow up'], errors='coerce')

# Count & drop NaNs
n_missing_initial = df_age_lf['Age last follow up'].isna().sum()
logger.info(f"{n_missing_initial} patients have no 'Age last follow up' recorded")

# Histogram
plt.figure(figsize=(8, 5))
plt.hist(df_age_lf['Age last follow up'].values, bins=10, color='skyblue', edgecolor='black')
plt.xticks(np.arange(0, df_age_lf['Age last follow up'].max() + 20, 20))
plt.xlabel('Age at Last Follow-Up (months)')
plt.ylabel('Number of Patients')
plt.suptitle('Infants group: Distribution of Age at Last Follow-Up')
plt.title('(Patients with recorded TUR age <= 12m)')
plt.tight_layout()
plt.show()